# 📅 Exercício Extra 1: O Seguidor de Linha Ótico Autónomo 🏎️🛣️

Neste exercício, vamos dar o derradeiro passo em condução autónoma tradicional. O teu JetRacer vai usar a câmara para encontrar uma linha no chão (ex: fita isoladora amarela ou preta) e **conduzir sozinho** ao longo dela!

### 📐 A Matemática por trás da Condução: Controlo Proporcional
1. O robô encontra o centro da linha na imagem usando a matemática dos **Momentos** (`cv2.moments`).
2. Ele calcula o **Erro** (a distância entre o centro do ecrã e o centro da linha).
3. O robô calcula o ângulo de viragem das rodas usando a fórmula:
   $$\text{Direção} = \text{Erro} \times K_{\text{proporcional}}$$

### 🎯 O Teu Desafio
Afinar o valor do **`K_proporcional`** (Ganho). 
* Se for muito baixo, o carro não reage a tempo e sai em frente nas curvas.
* Se for muito alto, o carro reage de forma violenta e começa a serpentear de um lado para o outro.

---

### 🛠️ Instruções Passo a Passo
1. ⚠️ **Segurança:** Faz os primeiros testes com as **rodas no ar**!
2. Executa esta célula para abrir os ecrãs e os controlos.
3. Ajusta os sliders `H Mínimo` e `H Máximo` até que apenas a fita do chão fique visível (com o centro marcado por uma linha vermelha).
4. Altera o valor de `K_proporcional` diretamente no código abaixo para encontrar o comportamento de curva perfeito!

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jethelper import mover, parar
import time

print("--- SEGUIDOR DE LINHA ÓTICO AUTÓNOMO ATIVO ---")

# ==========================================================
# ⚙️ PARÂMETROS DE CONDUÇÃO (Ajusta estes valores na aula!)
# ==========================================================
velocidade_base = 0.1# Velocidade constante de avanço (12%)
K_proporcional = 0.8 # Multiplicador de viragem (Ponto doce: 0.004 a 0.008)

# 1. Inicializar a câmara e criar os componentes visuais
camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)

slider_h_min = widgets.IntSlider(value=20, min=0, max=179, description='H Mínimo:')
slider_h_max = widgets.IntSlider(value=40, min=0, max=179, description='H Máximo:')
botao_desligar = widgets.Button(description="❌ DESLIGAR SISTEMA", button_style='danger')

display(imagem_widget)
display(widgets.VBox([slider_h_min, slider_h_max, botao_desligar]))

sistema_ativo = True

# 2. Função de Processamento de Imagem e Piloto Automático
def processar_linha_autonoma(change):
    global sistema_ativo
    if not sistema_ativo:
        parar()
        return
        
    frame = change['new']
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    baixo = np.array([0, 0, 0])
    alto = np.array([179, 60, 100])
    mascara = cv2.inRange(hsv, baixo, alto)
    
    # Calcular os Momentos geométricos da imagem filtrada
    M = cv2.moments(mascara)
    
    centro_imagem_x = 150 # Centro exato do ecrã (300 / 2)
    
    if M["m00"] > 0:
        # Encontra a coordenada X do centro da linha
        centro_linha_x = int(M["m10"] / M["m00"])
        
        # Calcula o desvio (Erro)
        erro = centro_linha_x - centro_imagem_x
        
        # --- CONTROLO PROPORCIONAL DA DIREÇÃO ---
        direcao = erro * K_proporcional
        
        # Garante que o valor enviado para a direção não ultrapassa os limites (-1.0 a 1.0)
        direcao = max(min(direcao, 1.0), -1.0)
        
        # ENVIAR ORDENS DE CONDUÇÃO PARA O HARDWARE
        mover(velocidade_base, direcao)
        
        print(f"ERRO: {erro:3d} px | DIREÇÃO CALCULADA: {direcao:5.2f} -> A Conduzir...", end='\r')
        
        # Desenhar guias visuais no ecrã do Jupyter
        cv2.line(frame, (centro_linha_x, 0), (centro_linha_x, 300), (0, 0, 255), 3) # Posição da linha
        cv2.line(frame, (centro_imagem_x, 0), (centro_imagem_x, 300), (255, 0, 0), 1) # Centro do carro
    else:
        # Se o carro perder a linha completamente de vista, pára por segurança
        print("⚠️ Linha perdida! A travar motores imediatamente...             ", end='\r')
        parar()
        
    # Atualiza a imagem com os gráficos visuais
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

# Conectar o fluxo da câmara à nossa função de condução autónoma
camera.observe(processar_linha_autonoma, names='value')

# 3. Botão para Desligar de Emergência
def desligar_tudo(b):
    global sistema_ativo
    sistema_ativo = False
    print("\n[A parar motores e a desligar câmara...]")
    parar()
    try:
        camera.unobserve(processar_linha_autonoma, names='value')
        camera.running = False
    except:
        pass
    botao_desligar.description = "🛑 PILOTO AUTOMÁTICO INATIVO"
    botao_desligar.button_style = "info"
    botao_desligar.disabled = True
    print("Plataforma imobilizada com sucesso!")

botao_desligar.on_click(desligar_tudo)
camera.running = True

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


--- SEGUIDOR DE LINHA ÓTICO AUTÓNOMO ATIVO ---


Image(value=b'', format='jpeg', height='300', width='300')